In [1]:
# %%
import os
import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt
import subprocess
import sys
import seaborn as sns
import numpy as np
from scipy.sparse import csr_matrix
import scanpy.external as sce
from sklearn.metrics import silhouette_score
import datetime
from collections import defaultdict
import scipy.sparse as sp
import gffutils 

# Configuration
%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}
%config InlineBackend.figure_format='retina'

# Get current directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

# Global flags and paths
read_introns_exons = True
today = datetime.datetime.now().strftime("%Y-%m-%d")

# File paths
gtf_file = "/gpfs/commons/groups/knowles_lab/Megan/encode_pacbio/paper_figures/isoform_gazers/all_samples_sp_collapse_all_chr_no_treatment_hashid_isoform_full.gtf"
db_file = "/gpfs/commons/groups/knowles_lab/Megan/encode_pacbio/paper_figures/isoform_gazers/long_read_hg38.db"
WD = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression"
output_dir = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data"

# %%
def validate_file_exists(filepath, description=""):
    """Validate that a file exists and is readable."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"❌ {description} file not found: {filepath}")
    if not os.access(filepath, os.R_OK):
        raise PermissionError(f"❌ {description} file not readable: {filepath}")
    print(f"✅ {description} file found: {filepath}")
    return True

def extract_gene_transcript_info(gtf_file, db_file):
    """
    Parses a GENCODE GTF file to compute gene transcript information.
    
    Returns:
        DataFrame with gene_id, gene_name, mean_transcript_length, mean_intron_length, 
        num_transcripts, and transcript_biotypes.
    """
    print("\n=== EXTRACTING GENE TRANSCRIPT INFORMATION ===")
    
    # SANITY CHECK 1: Validate input files
    validate_file_exists(gtf_file, "GTF")
    
    # Load or create database
    if os.path.exists(db_file):
        print("✅ Using existing GTF database")
        db = gffutils.FeatureDB(db_file, keep_order=True)
        print("✅ Database loaded successfully!")
    else:
        print("⏳ Creating GTF database (this may take a few minutes)...")
        validate_file_exists(gtf_file, "GTF")
        db = gffutils.create_db(
            gtf_file,
            db_file,
            force=True,
            keep_order=True,
            disable_infer_transcripts=False,
            disable_infer_genes=True
        )
        print("✅ Database created successfully!")

    # Initialize data structures
    gene_exon_lengths = defaultdict(list)
    gene_intron_lengths = defaultdict(list)
    gene_names = {}
    gene_biotypes = defaultdict(set)
    transcript_counts = defaultdict(int)
    
    # Counters for sanity checks
    total_transcripts = 0
    skipped_transcripts = 0

    print("⏳ Processing transcripts to compute exon and intron lengths...")
    
    for transcript in tqdm(db.features_of_type("transcript"), desc="Processing Transcripts"):
        total_transcripts += 1
        
        # Extract transcript attributes
        gene_id = transcript.attributes.get("gene_id", [None])[0]
        gene_name = transcript.attributes.get("gene_name", ["unknown"])[0]
        transcript_biotype = transcript.attributes.get("transcript_type", ["unknown"])[0]
        
        if gene_id is None:
            print(f"⚠️  Skipping transcript without gene_id: {transcript.id}")
            skipped_transcripts += 1
            continue

        # Get exons for this transcript
        exons = list(db.children(transcript, featuretype="exon", order_by="start"))
        if len(exons) == 0:
            skipped_transcripts += 1
            continue

        # Calculate lengths
        exon_length = sum(exon.end - exon.start + 1 for exon in exons)
        transcript_start = min(exon.start for exon in exons)
        transcript_end = max(exon.end for exon in exons)
        transcript_span = transcript_end - transcript_start + 1
        intron_length = transcript_span - exon_length

        # Store data
        gene_exon_lengths[gene_id].append(exon_length)
        gene_intron_lengths[gene_id].append(max(0, intron_length))
        gene_names[gene_id] = gene_name
        gene_biotypes[gene_id].add(transcript_biotype)
        transcript_counts[gene_id] += 1

    # SANITY CHECK 2: Processing summary
    print(f"\n=== TRANSCRIPT PROCESSING SUMMARY ===")
    print(f"Total transcripts processed: {total_transcripts:,}")
    print(f"Transcripts skipped (no exons/gene_id): {skipped_transcripts:,}")
    print(f"Transcripts used: {total_transcripts - skipped_transcripts:,}")
    print(f"Unique genes found: {len(gene_exon_lengths):,}")

    if len(gene_exon_lengths) == 0:
        raise ValueError("❌ No valid genes found in GTF file!")

    # Create final DataFrame
    gene_ids = list(gene_exon_lengths.keys())
    gene_info_df = pd.DataFrame({
        "gene_id": gene_ids,
        "gene_name": [gene_names[g] for g in gene_ids],
        "mean_transcript_length": [sum(gene_exon_lengths[g]) / len(gene_exon_lengths[g]) for g in gene_ids],
        "mean_intron_length": [sum(gene_intron_lengths[g]) / len(gene_intron_lengths[g]) for g in gene_ids],
        "num_transcripts": [transcript_counts[g] for g in gene_ids],
        "transcript_biotypes": [", ".join(sorted(gene_biotypes[g])) for g in gene_ids]
    })

    # SANITY CHECK 3: Final DataFrame validation
    print(f"\n=== GENE INFO VALIDATION ===")
    print(f"Gene info DataFrame shape: {gene_info_df.shape}")
    print(f"Columns: {list(gene_info_df.columns)}")
    
    # Check for missing values
    missing_counts = gene_info_df.isnull().sum()
    if missing_counts.any():
        print(f"⚠️  Missing values found:\n{missing_counts[missing_counts > 0]}")
    else:
        print("✅ No missing values in gene info")
    
    # Basic statistics
    print(f"Mean transcript length range: {gene_info_df['mean_transcript_length'].min():.0f} - {gene_info_df['mean_transcript_length'].max():.0f}")
    print(f"Mean intron length range: {gene_info_df['mean_intron_length'].min():.0f} - {gene_info_df['mean_intron_length'].max():.0f}")
    print(f"Transcript count range: {gene_info_df['num_transcripts'].min()} - {gene_info_df['num_transcripts'].max()}")
    
    # Check for duplicates
    gene_id_dups = gene_info_df['gene_id'].duplicated().sum()
    gene_name_dups = gene_info_df['gene_name'].duplicated().sum()
    print(f"Duplicate gene_ids: {gene_id_dups}")
    print(f"Duplicate gene_names: {gene_name_dups}")
    
    if gene_id_dups > 0 or gene_name_dups > 0:
        print("⚠️  WARNING: Duplicates found in gene info!")

    return gene_info_df

def preprocess_ab_adata(adata, metadata, dataset_label="allen_brain", 
                       metadata_key="sample_name", rename_var=True):
    """
    Standardizes Allen Brain adata object with comprehensive validation.
    """
    print(f"\n=== PREPROCESSING {dataset_label.upper()} DATA ===")
    
    # SANITY CHECK 1: Input validation
    print(f"Input adata shape: {adata.shape}")
    print(f"Metadata shape: {metadata.shape}")
    print(f"Metadata key: {metadata_key}")
    
    if metadata_key not in metadata.columns:
        raise ValueError(f"❌ Metadata key '{metadata_key}' not found in columns: {list(metadata.columns)}")
    
    # Create copy to avoid modifying original
    adata = adata.copy()
    
    # Basic setup
    adata.var['gene_name'] = adata.var_names
    adata.obs["dataset"] = dataset_label

    # SANITY CHECK 2: Metadata overlap
    adata_cells = set(adata.obs_names)
    metadata_cells = set(metadata[metadata_key])
    overlap = adata_cells.intersection(metadata_cells)
    
    print(f"Cells in adata: {len(adata_cells):,}")
    print(f"Cells in metadata: {len(metadata_cells):,}")
    print(f"Overlapping cells: {len(overlap):,}")
    
    if len(overlap) == 0:
        raise ValueError("❌ No overlapping cells between adata and metadata!")
    
    if len(overlap) < len(adata_cells) * 0.8:
        print(f"⚠️  WARNING: Only {len(overlap)/len(adata_cells)*100:.1f}% of cells have metadata")
    
    # Subset and align metadata
    metadata_sub = metadata[metadata[metadata_key].isin(adata.obs_names)].copy()
    
    # SANITY CHECK 3: Check for duplicate metadata entries
    if metadata_sub[metadata_key].duplicated().any():
        dup_count = metadata_sub[metadata_key].duplicated().sum()
        print(f"⚠️  WARNING: {dup_count} duplicate entries in metadata - keeping first occurrence")
        metadata_sub = metadata_sub.drop_duplicates(subset=metadata_key, keep='first')
    
    metadata_sub = metadata_sub.set_index(metadata_key)

    # Align adata to metadata
    adata = adata[adata.obs_names.isin(metadata_sub.index)].copy()
    
    # SANITY CHECK 4: Verify alignment
    if adata.shape[0] != len(metadata_sub):
        print(f"⚠️  Shape mismatch: adata {adata.shape[0]} vs metadata {len(metadata_sub)}")
    
    # Check index alignment
    if not all(adata.obs_names.isin(metadata_sub.index)):
        missing_cells = set(adata.obs_names) - set(metadata_sub.index)
        print(f"❌ ERROR: {len(missing_cells)} cells in adata not found in metadata")
        raise ValueError("Cell alignment failed!")
    
    # Apply metadata
    adata.obs = metadata_sub.loc[adata.obs_names]
    
    # SANITY CHECK 5: Post-alignment validation
    print(f"Final aligned shape: {adata.shape}")
    print(f"Metadata columns added: {len(metadata_sub.columns)}")
    
    # Handle gene naming
    if rename_var:
        if "gene_name" in adata.var.columns:
            adata.var.rename(columns={"gene_name": "gene_symbol"}, inplace=True)
        else:
            print("⚠️  WARNING: 'gene_name' column not found for renaming")
    else:
        adata.var["gene_symbol"] = adata.var["gene_name"]
    
    # Store raw counts
    if sp.issparse(adata.X):
        adata.layers["raw_counts"] = adata.X.copy()
    else:
        adata.layers["raw_counts"] = sp.csr_matrix(adata.X)
    
    # SANITY CHECK 6: Expression data validation
    print(f"Expression matrix type: {type(adata.X)}")
    if sp.issparse(adata.X):
        print(f"Sparse format: {adata.X.format}")
        print(f"Non-zero elements: {adata.X.nnz:,}")
        min_val = adata.X.min()
        max_val = adata.X.max()
    else:
        min_val = adata.X.min()
        max_val = adata.X.max()
    
    print(f"Expression range: {min_val} - {max_val}")
    
    if min_val < 0:
        print(f"⚠️  WARNING: Negative values found in expression data!")
    
    print(f"✅ Successfully processed {dataset_label}: {adata.shape[0]:,} cells, {adata.shape[1]:,} genes")
    return adata

# %%
print("=== STARTING DATA PROCESSING PIPELINE ===")

# SANITY CHECK: Validate all input files exist
print("\n=== FILE VALIDATION ===")
validate_file_exists(gtf_file, "GTF")

# Extract gene information
gene_info_df = extract_gene_transcript_info(gtf_file, db_file)

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis/Human_Splicing_Foundation/GeneExpression
=== STARTING DATA PROCESSING PIPELINE ===

=== FILE VALIDATION ===
✅ GTF file found: /gpfs/commons/groups/knowles_lab/Megan/encode_pacbio/paper_figures/isoform_gazers/all_samples_sp_collapse_all_chr_no_treatment_hashid_isoform_full.gtf

=== EXTRACTING GENE TRANSCRIPT INFORMATION ===
✅ GTF file found: /gpfs/commons/groups/knowles_lab/Megan/encode_pacbio/paper_figures/isoform_gazers/all_samples_sp_collapse_all_chr_no_treatment_hashid_isoform_full.gtf
✅ Using existing GTF database
✅ Database loaded successfully!
⏳ Processing transcripts to compute exon and intron lengths...


Processing Transcripts: 199406it [01:07, 2972.98it/s]


=== TRANSCRIPT PROCESSING SUMMARY ===
Total transcripts processed: 199,406
Transcripts skipped (no exons/gene_id): 0
Transcripts used: 199,406
Unique genes found: 19,226

=== GENE INFO VALIDATION ===
Gene info DataFrame shape: (19226, 6)
Columns: ['gene_id', 'gene_name', 'mean_transcript_length', 'mean_intron_length', 'num_transcripts', 'transcript_biotypes']
✅ No missing values in gene info
Mean transcript length range: 107 - 16120
Mean intron length range: 20 - 948051
Transcript count range: 1 - 316
Duplicate gene_ids: 0
Duplicate gene_names: 19225
⚠️  WARNING: Duplicates found in gene info!


In [2]:
import mygene

# Initialize mygene
mg = mygene.MyGeneInfo()

# Create a clean gene_id column without version numbers
gene_info_df['gene_id_clean'] = gene_info_df['gene_id'].str.replace(r'\.\d+$', '', regex=True)

# Query gene info - request the gene biotype field
gene_ids = gene_info_df['gene_id_clean'].tolist()
results = mg.querymany(gene_ids, 
                       scopes='ensembl.gene', 
                       fields='symbol,type_of_gene',  # type_of_gene gives you the gene biotype
                       species='human', 
                       returnall=True)

# Process results
gene_info = pd.DataFrame(results['out'])

# Merge back to original dataframe
df_merged = gene_info_df.merge(
    gene_info[['query', 'symbol', 'type_of_gene']], 
    left_on='gene_id_clean',
    right_on='query', 
    how='left'
)

# Rename columns for clarity
df_merged = df_merged.rename(columns={
    'symbol': 'gene_name_new',
    'type_of_gene': 'gene_biotype'
})

# Drop the temporary columns if desired
df_merged = df_merged.drop(columns=['gene_id_clean', 'query'])

# Update the gene_name column with the new data where available
df_merged['gene_name'] = df_merged['gene_name_new'].fillna(df_merged['gene_name'])
df_merged = df_merged.drop(columns=['gene_name_new'])

print(df_merged.shape)
print(df_merged.head())

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
3 input query terms found dup hits:	[('ENSG00000291072', 2), ('ENSG00000227110', 2), ('ENSG00000175711', 2)]
1961 input query terms found no hit:	['chr1:1325000', 'chr1:1435000', 'chr1:9293000', 'chr1:9943000', 'chr1:16888000', 'chr1:19644000', '


(19229, 7)
              gene_id  gene_name  mean_transcript_length  mean_intron_length  \
0  ENSG00000228794.12  LINC01128             5155.611111        25435.388889   
1  ENSG00000187634.13     SAMD11             3502.000000        17161.500000   
2  ENSG00000188976.11      NOC2L             2946.500000        12161.516129   
3  ENSG00000187961.15     KLHL17             2887.470588         2245.058824   
4  ENSG00000188290.11       HES4             1963.583333          207.916667   

   num_transcripts transcript_biotypes    gene_biotype  
0               18             unknown           ncRNA  
1                4             unknown  protein-coding  
2               62             unknown  protein-coding  
3               17             unknown  protein-coding  
4               12             unknown  protein-coding  


In [3]:
# Let's remove any genes whose gene_name is unknown 
df_merged = df_merged[df_merged['gene_name'] != 'unknown']

# Remove transcript_biotype column
df_merged = df_merged.drop(columns=['transcript_biotypes'])

# remove genes that are not protein coding
df_merged = df_merged[df_merged['gene_biotype'] == 'protein-coding']
gene_info_df = df_merged.copy() 

In [4]:
# Remove duplicates with validation
print("\n=== CLEANING GENE INFO ===")
initial_shape = gene_info_df.shape
gene_info_df = gene_info_df.drop_duplicates(subset="gene_id")
after_gene_id_dedup = gene_info_df.shape
gene_info_df = gene_info_df.drop_duplicates(subset="gene_name")
final_shape = gene_info_df.shape

print(f"Gene info cleaning:")
print(f"  Initial: {initial_shape}")
print(f"  After gene_id dedup: {after_gene_id_dedup}")
print(f"  After gene_name dedup: {final_shape}")
print(f"  Rows removed: {initial_shape[0] - final_shape[0]}")


=== CLEANING GENE INFO ===
Gene info cleaning:
  Initial: (15217, 6)
  After gene_id dedup: (15216, 6)
  After gene_name dedup: (15208, 6)
  Rows removed: 9


In [5]:
gene_info_df

,gene_id,gene_name,mean_transcript_length,mean_intron_length,num_transcripts,gene_biotype
1,ENSG00000187634.13,SAMD11,3502.000000,17161.500000,4,protein-coding
2,ENSG00000188976.11,NOC2L,2946.500000,12161.516129,62,protein-coding
3,ENSG00000187961.15,KLHL17,2887.470588,2245.058824,17,protein-coding
4,ENSG00000188290.11,HES4,1963.583333,207.916667,12,protein-coding
5,ENSG00000187608.10,ISG15,637.000000,407.000000,1,protein-coding
...,...,...,...,...,...,...
19217,ENSG00000183878.16,UTY,5415.750000,207562.500000,4,protein-coding
19218,ENSG00000154620.6,TMSB4Y,1336.000000,789.000000,1,protein-coding
19219,ENSG00000165246.15,NLGN4Y,6715.333333,314475.000000,6,protein-coding
19222,ENSG00000012817.16,KDM5D,6256.666667,34086.833333,6,protein-coding


In [6]:
# %%
print("\n=== PROCESSING ALLEN BRAIN DATA ===")

# Validate Allen Brain input files
introns_only = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/intron.csv"
exons_only = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/exon.csv"
metadata_file = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/INFO/metadata.csv"

if read_introns_exons:
    validate_file_exists(introns_only, "Introns")
    validate_file_exists(exons_only, "Exons")
validate_file_exists(metadata_file, "Metadata")

# Load Allen Brain data
processing_stats = {"datasets_loaded": 0, "datasets_processed": 0}

if read_introns_exons:
    print("\n--- Loading Introns Data ---")
    ab_adata_introns = sc.read_csv(introns_only)
    ab_adata_introns = ab_adata_introns.transpose()
    print(f"✅ Introns shape: {ab_adata_introns.shape}")
    processing_stats["datasets_loaded"] += 1

    print("\n--- Loading Exons Data ---")
    ab_adata_exons = sc.read_csv(exons_only)
    ab_adata_exons = ab_adata_exons.transpose()
    print(f"✅ Exons shape: {ab_adata_exons.shape}")
    processing_stats["datasets_loaded"] += 1

# Load metadata
print("\n--- Loading Metadata ---")
metadata = pd.read_csv(metadata_file)
print(f"✅ Metadata shape: {metadata.shape}")
print(f"Metadata columns: {list(metadata.columns)}")

# SANITY CHECK: Verify expected metadata key exists
expected_key = "exp_component_name"
if expected_key not in metadata.columns:
    print(f"⚠️  WARNING: Expected metadata key '{expected_key}' not found")
    print(f"Available keys: {list(metadata.columns)}")

# Process Allen Brain datasets
if read_introns_exons:
    ab_adata_introns = preprocess_ab_adata(
        ab_adata_introns, metadata, 
        dataset_label="allen_brain_introns",
        metadata_key="exp_component_name"
    )
    processing_stats["datasets_processed"] += 1
    
    ab_adata_exons = preprocess_ab_adata(
        ab_adata_exons, metadata, 
        dataset_label="allen_brain_exons",
        metadata_key="exp_component_name"
    )
    processing_stats["datasets_processed"] += 1

print(f"✅ Allen Brain processing complete: {processing_stats}")


=== PROCESSING ALLEN BRAIN DATA ===
✅ Introns file found: /gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/intron.csv
✅ Exons file found: /gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/exon.csv
✅ Metadata file found: /gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/INFO/metadata.csv

--- Loading Introns Data ---
✅ Introns shape: (49493, 50281)

--- Loading Exons Data ---
✅ Exons shape: (49493, 50281)

--- Loading Metadata ---
✅ Metadata shape: (49417, 41)
Metadata columns: ['sample_name', 'exp_component_name', 'specimen_type', 'cluster_color', 'cluster_order', 'cluster_label', 'class_color', 'class_order', 'class_label', 'subclass_color', 'subclass_order', 'subclass_label', 'full_genotype_color', 'full_genotype_order', 'full_genotype_label', 'donor_sex_color', 'donor_sex_order', 'donor_sex_label', 'region_color', 'region_order', 'region_label', 'cortical_layer_color', 'cortical_layer_order', 'cortical_layer_labe

In [7]:
# %%
print("\n=== PROCESSING TABULA SAPIENS DATA ===")

# Load Tabula Sapiens
tabsap_file = "/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/merged_tabula_sapiens.h5ad"
validate_file_exists(tabsap_file, "Tabula Sapiens")

print("⏳ Loading Tabula Sapiens V2 data...")
tabsap_adata = sc.read_h5ad(tabsap_file)
tabsap_adata.obs["dataset"] = "tabula_sapiens"

print(f"✅ Loaded Tabula Sapiens: {tabsap_adata.shape}")

# SANITY CHECK 1: Verify this is SS2 data only
assay_counts = tabsap_adata.obs["assay"].value_counts()
print(f"Assay distribution:\n{assay_counts}")

if len(assay_counts) > 1 or "SS2" not in assay_counts.index:
    print("⚠️  WARNING: Non-SS2 data found in Tabula Sapiens!")
else:
    print("✅ All cells are SS2 as expected")

# SANITY CHECK 2: Handle gene symbol duplicates
print("\n--- Handling Gene Duplicates ---")
initial_genes = tabsap_adata.shape[1]
gene_symbol_dups = tabsap_adata.var["gene_symbol"].duplicated().sum()
print(f"Duplicate gene symbols: {gene_symbol_dups}")

if gene_symbol_dups > 0:
    print("⚠️  Removing duplicate gene symbols...")
    duplicated_genes = tabsap_adata.var[tabsap_adata.var["gene_symbol"].duplicated(keep=False)]
    print(f"Genes to remove: {len(duplicated_genes)}")
    
    # Show some examples
    if len(duplicated_genes) > 0:
        print(f"Example duplicates: {duplicated_genes['gene_symbol'].head().tolist()}")
    
    tabsap_adata = tabsap_adata[:, ~tabsap_adata.var.index.isin(duplicated_genes.index)].copy()
    final_genes = tabsap_adata.shape[1]
    print(f"Genes removed: {initial_genes - final_genes}")
    
    # Verify no duplicates remain
    remaining_dups = tabsap_adata.var["gene_symbol"].duplicated().sum()
    if remaining_dups > 0:
        print(f"❌ ERROR: {remaining_dups} duplicates still remain!")
    else:
        print("✅ All duplicates removed")
else:
    print("✅ No gene symbol duplicates found")

# Set up gene naming consistency
tabsap_adata.var["gene_name"] = tabsap_adata.var["gene_symbol"]


=== PROCESSING TABULA SAPIENS DATA ===
✅ Tabula Sapiens file found: /gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/merged_tabula_sapiens.h5ad
⏳ Loading Tabula Sapiens V2 data...
✅ Loaded Tabula Sapiens: (41501, 61806)
Assay distribution:
assay
SS2    41501
Name: count, dtype: int64
✅ All cells are SS2 as expected

--- Handling Gene Duplicates ---
Duplicate gene symbols: 1200
⚠️  Removing duplicate gene symbols...
Genes to remove: 1277
Example duplicates: ['5S_rRNA', '5S_rRNA', '5S_rRNA', '5S_rRNA', '5S_rRNA']
Genes removed: 1277
✅ All duplicates removed


In [8]:
# SANITY CHECK 3: Filter to genes in gene_info_df
print("\n--- Filtering to Known Genes ---")
tabsap_genes_before = tabsap_adata.shape[1]
known_genes = set(gene_info_df["gene_name"])
tabsap_genes_in_db = set(tabsap_adata.var["gene_name"]).intersection(known_genes)

print(f"Tabula Sapiens genes: {len(tabsap_adata.var['gene_name']):,}")
print(f"Genes in gene_info_df: {len(known_genes):,}")
print(f"Overlapping genes: {len(tabsap_genes_in_db):,}")

tabsap_adata = tabsap_adata[:, tabsap_adata.var["gene_name"].isin(known_genes)].copy()
tabsap_genes_after = tabsap_adata.shape[1]

print(f"Genes filtered out: {tabsap_genes_before - tabsap_genes_after}")
print(f"Final gene count: {tabsap_genes_after:,}")

# Store raw counts
tabsap_adata.layers["raw_counts"] = tabsap_adata.X.copy()


--- Filtering to Known Genes ---
Tabula Sapiens genes: 60,529
Genes in gene_info_df: 15,208
Overlapping genes: 15,015
Genes filtered out: 45514
Final gene count: 15,015


In [9]:
# SANITY CHECK 4: Cross-dataset gene overlap
if read_introns_exons:
    print("\n--- Cross-Dataset Gene Overlap ---")
    tabsap_genes = set(tabsap_adata.var["gene_symbol"])
    ab_exon_genes = set(ab_adata_exons.var["gene_symbol"])
    ab_intron_genes = set(ab_adata_introns.var["gene_symbol"])
    
    tabsap_ab_overlap = tabsap_genes.intersection(ab_exon_genes)
    print(f"Tabula Sapiens genes: {len(tabsap_genes):,}")
    print(f"Allen Brain exon genes: {len(ab_exon_genes):,}")
    print(f"Allen Brain intron genes: {len(ab_intron_genes):,}")
    print(f"Tabula Sapiens ↔ Allen Brain overlap: {len(tabsap_ab_overlap):,}")
    
    if len(tabsap_ab_overlap) < min(len(tabsap_genes), len(ab_exon_genes)) * 0.5:
        print("⚠️  WARNING: Low gene overlap between datasets!")
    else:
        print("✅ Good gene overlap between datasets")


--- Cross-Dataset Gene Overlap ---
Tabula Sapiens genes: 15,015
Allen Brain exon genes: 50,281
Allen Brain intron genes: 50,281
Tabula Sapiens ↔ Allen Brain overlap: 14,277
✅ Good gene overlap between datasets


In [10]:
print("\n=== PREPARING DATA FOR SAVING ===")

# Convert to sparse matrices
if read_introns_exons:
    print("Converting Allen Brain data to sparse format...")
    if not sp.issparse(ab_adata_exons.X):
        ab_adata_exons.X = csr_matrix(ab_adata_exons.X)
        print("✅ Converted exons to sparse")
    else:
        print("✅ Exons already sparse")
        
    if not sp.issparse(ab_adata_introns.X):
        ab_adata_introns.X = csr_matrix(ab_adata_introns.X)
        print("✅ Converted introns to sparse")
    else:
        print("✅ Introns already sparse")

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Output directory ready: {output_dir}")

# %%
print("\n=== SAVING PROCESSED DATA ===")

save_stats = {"files_saved": 0, "total_size_mb": 0}

try:
    # Save Tabula Sapiens
    tabsap_path = os.path.join(output_dir, f"tabsap_adata_{today}.h5ad")
    print(f"💾 Saving Tabula Sapiens to: {tabsap_path}")
    tabsap_adata.write_h5ad(tabsap_path, compression="lzf")
    tabsap_size = os.path.getsize(tabsap_path) / (1024**2)  # MB
    print(f"✅ Tabula Sapiens saved ({tabsap_size:.1f} MB)")
    save_stats["files_saved"] += 1
    save_stats["total_size_mb"] += tabsap_size

    if read_introns_exons:
        # Save Allen Brain exons
        exons_path = os.path.join(output_dir, f"ab_adata_exons_{today}.h5ad")
        print(f"💾 Saving Allen Brain exons to: {exons_path}")
        ab_adata_exons.write_h5ad(exons_path, compression="lzf")
        exons_size = os.path.getsize(exons_path) / (1024**2)
        print(f"✅ Allen Brain exons saved ({exons_size:.1f} MB)")
        save_stats["files_saved"] += 1
        save_stats["total_size_mb"] += exons_size

        # Save Allen Brain introns
        introns_path = os.path.join(output_dir, f"ab_adata_introns_{today}.h5ad")
        print(f"💾 Saving Allen Brain introns to: {introns_path}")
        ab_adata_introns.write_h5ad(introns_path, compression="lzf")
        introns_size = os.path.getsize(introns_path) / (1024**2)
        print(f"✅ Allen Brain introns saved ({introns_size:.1f} MB)")
        save_stats["files_saved"] += 1
        save_stats["total_size_mb"] += introns_size

    # Save gene info
    gene_info_path = os.path.join(output_dir, f"gene_info_df_{today}.csv")
    print(f"💾 Saving gene info to: {gene_info_path}")
    gene_info_df.to_csv(gene_info_path, index=False)
    gene_info_size = os.path.getsize(gene_info_path) / (1024**2)
    print(f"✅ Gene info saved ({gene_info_size:.1f} MB)")
    save_stats["files_saved"] += 1
    save_stats["total_size_mb"] += gene_info_size

except Exception as e:
    print(f"❌ ERROR during saving: {e}")
    raise

print(f"\n🎉 SAVE SUMMARY:")
print(f"Files saved: {save_stats['files_saved']}")
print(f"Total size: {save_stats['total_size_mb']:.1f} MB")


=== PREPARING DATA FOR SAVING ===
Converting Allen Brain data to sparse format...
✅ Converted exons to sparse
✅ Converted introns to sparse
✅ Output directory ready: /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data

=== SAVING PROCESSED DATA ===
💾 Saving Tabula Sapiens to: /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data/tabsap_adata_2025-09-30.h5ad
✅ Tabula Sapiens saved (768.0 MB)
💾 Saving Allen Brain exons to: /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data/ab_adata_exons_2025-09-30.h5ad
✅ Allen Brain exons saved (2671.8 MB)
💾 Saving Allen Brain introns to: /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data/ab_adata_introns_2025-09-30.h5ad
✅ Allen Brain introns saved (3385.0 MB)
💾 Saving gene info to: /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATI

In [11]:
# %%
print("\n=== POST-SAVE VERIFICATION ===")

# Verify saved files can be read back
verification_passed = 0
total_verifications = save_stats["files_saved"]

try:
    # Verify Tabula Sapiens
    print("🔍 Verifying Tabula Sapiens...")
    test_tabsap = sc.read_h5ad(tabsap_path)
    if test_tabsap.shape == tabsap_adata.shape:
        print("✅ Tabula Sapiens verification passed")
        verification_passed += 1
    else:
        print(f"❌ Shape mismatch: saved {test_tabsap.shape} vs original {tabsap_adata.shape}")
    del test_tabsap

    if read_introns_exons:
        # Verify Allen Brain exons
        print("🔍 Verifying Allen Brain exons...")
        test_exons = sc.read_h5ad(exons_path)
        if test_exons.shape == ab_adata_exons.shape:
            print("✅ Allen Brain exons verification passed")
            verification_passed += 1
        else:
            print(f"❌ Shape mismatch: saved {test_exons.shape} vs original {ab_adata_exons.shape}")
        del test_exons

        # Verify Allen Brain introns
        print("🔍 Verifying Allen Brain introns...")
        test_introns = sc.read_h5ad(introns_path)
        if test_introns.shape == ab_adata_introns.shape:
            print("✅ Allen Brain introns verification passed")
            verification_passed += 1
        else:
            print(f"❌ Shape mismatch: saved {test_introns.shape} vs original {ab_adata_introns.shape}")
        del test_introns

    # Verify gene info
    print("🔍 Verifying gene info...")
    test_gene_info = pd.read_csv(gene_info_path)
    if test_gene_info.shape == gene_info_df.shape:
        print("✅ Gene info verification passed")
        verification_passed += 1
    else:
        print(f"❌ Shape mismatch: saved {test_gene_info.shape} vs original {gene_info_df.shape}")

except Exception as e:
    print(f"❌ Verification failed: {e}")

print(f"\n📋 VERIFICATION SUMMARY: {verification_passed}/{total_verifications} files verified")

# %%
print("\n=== FINAL PROCESSING SUMMARY ===")
print("🎉 DATA PROCESSING PIPELINE COMPLETE!")
print(f"📅 Processing date: {today}")
print(f"📁 Output directory: {output_dir}")
print(f"🧬 Gene info records: {len(gene_info_df):,}")
print(f"🔬 Tabula Sapiens: {tabsap_adata.shape[0]:,} cells, {tabsap_adata.shape[1]:,} genes")

if read_introns_exons:
    print(f"🧠 Allen Brain exons: {ab_adata_exons.shape[0]:,} cells, {ab_adata_exons.shape[1]:,} genes")
    print(f"🧠 Allen Brain introns: {ab_adata_introns.shape[0]:,} cells, {ab_adata_introns.shape[1]:,} genes")

print("\n✅ All processing complete with comprehensive validation!")

# Quick example of loading saved data back
print("\n=== EXAMPLE: LOADING SAVED DATA ===")
print("# To load the processed data later, use:")
print(f"gene_info_df = pd.read_csv('{gene_info_path}')")
print(f"tabsap_adata = sc.read_h5ad('{tabsap_path}')")
if read_introns_exons:
    print(f"ab_adata_exons = sc.read_h5ad('{exons_path}')")
    print(f"ab_adata_introns = sc.read_h5ad('{introns_path}')")

# %%
# Optional: Display sample of the gene info data
print("\n=== GENE INFO SAMPLE ===")
print("First 5 rows of gene_info_df:")
print(gene_info_df.head())
print(f"\nGene info statistics:")
print(gene_info_df.describe())

# %%
# Optional: Display variable information for verification
if read_introns_exons:
    print("\n=== DATASET VARIABLE COMPARISON ===")
    print("Allen Brain exons variables (first 5):")
    print(ab_adata_exons.var.head())
    print(f"\nAllen Brain exons var columns: {list(ab_adata_exons.var.columns)}")
    
    print("\nAllen Brain introns variables (first 5):")
    print(ab_adata_introns.var.head())
    print(f"\nAllen Brain introns var columns: {list(ab_adata_introns.var.columns)}")

print("\nTabula Sapiens variables (first 5):")
print(tabsap_adata.var.head())
print(f"\nTabula Sapiens var columns: {list(tabsap_adata.var.columns)}")

print(f"\n📋 PROCESSING COMPLETE - {today}")
print("=" * 60)


=== POST-SAVE VERIFICATION ===
🔍 Verifying Tabula Sapiens...
✅ Tabula Sapiens verification passed
🔍 Verifying Allen Brain exons...
✅ Allen Brain exons verification passed
🔍 Verifying Allen Brain introns...
✅ Allen Brain introns verification passed
🔍 Verifying gene info...
✅ Gene info verification passed

📋 VERIFICATION SUMMARY: 4/4 files verified

=== FINAL PROCESSING SUMMARY ===
🎉 DATA PROCESSING PIPELINE COMPLETE!
📅 Processing date: 2025-09-30
📁 Output directory: /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data
🧬 Gene info records: 15,208
🔬 Tabula Sapiens: 41,501 cells, 15,015 genes
🧠 Allen Brain exons: 49,417 cells, 50,281 genes
🧠 Allen Brain introns: 49,417 cells, 50,281 genes

✅ All processing complete with comprehensive validation!

=== EXAMPLE: LOADING SAVED DATA ===
# To load the processed data later, use:
gene_info_df = pd.read_csv('/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed